In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences
from dataloaders._ts_dataloader      import DataLoaderFactory, FullSeriesDataset
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common._base_model import BaseModel
from common.train import train, train_distributed, eval_test
from models.linear import TinyLinearModel

warnings.filterwarnings("ignore")
print("imports OK")

In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    horizon:           int  = 6,
    n_channels:        int  = 3, 
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        horizon                = horizon,
        n_channels             = n_channels, 
        context_length         = context_length,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
        loss                   = 'mae', 
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

## Test — `train()`, single dataset, concat mixing (default)

In [ ]:
print("=" * 60)
print("TEST 1 — single dataset, concat mixing")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    df = make_df(n_series=3, n_steps=300)
    df.to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length = 32,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        mixing_strategy= "concat",
        checkpoint_dir = tmpdir,
    )
    entry = make_entry(csv_path, name="test", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    model   = TinyLinearModel(mcfg)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"\nfinal val metrics: {metrics}")
    assert Path(tmpdir, "final.pt").exists(), "final.pt not saved"

    # Check predictions
    test_loaders = factory.test_dataloaders()
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "test" in preds, "missing test predictions"

print("\n✓ TEST PASSED")

## Test — `train()`, two datasets, weighted concat mixing

In [ ]:
print("=" * 60)
print("TEST 2 — two datasets, weighted concat mixing")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_a = f"{tmpdir}/data_a.csv"
    csv_b = f"{tmpdir}/data_b.csv"
    make_df(n_series=3, n_steps=300, seed=1).to_csv(csv_a, index=False)
    make_df(n_series=2, n_steps=300, seed=2).to_csv(csv_b, index=False)

    mcfg = make_mcfg(
        context_length  = 32,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        mixing_strategy = "concat",
        checkpoint_dir  = tmpdir,
    )

    # Same horizon, different weights — dataset A gets 3x more samples than B
    entry_a = make_entry(csv_a, name="ds_a", horizon=6, val_size=50, test_size=50, weight=3.0)
    entry_b = make_entry(csv_b, name="ds_b", horizon=6, val_size=50, test_size=50, weight=1.0)
    dcfg    = make_dcfg([entry_a, entry_b])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    # Max channels across both datasets = 3 (ds_a)
    model   = TinyLinearModel(mcfg)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"final val metrics: {metrics}")

    # Verify both datasets appear in val/test
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "ds_a" in preds and "ds_b" in preds, f"missing keys in preds: {preds.keys()}"

print("\n✓ TEST PASSED")

In [ ]:
# %%
import numpy as np
import torch
from collections import defaultdict
from torch.utils.data import ConcatDataset, DataLoader

from ts_dataloader import FullSeriesDataset, HorizonBatchSampler, _full_series_collate_fn
from config import _dataset_entry

# %%
# ── Helpers ──────────────────────────────────────────────────────────────────

def _make_dataset(n_channels, length=200, horizon=24, context_length=96,
                  is_multivariate=False, name="test"):
    rng = np.random.default_rng(42)
    y              = rng.random((length, n_channels)).astype(np.float32)
    hist           = np.zeros((length, n_channels, 0), dtype=np.float32)
    futr           = np.zeros((length, n_channels, 0), dtype=np.float32)
    available_mask = np.ones((length, n_channels), dtype=np.float32)
    return FullSeriesDataset(
        y               = y,
        hist            = hist,
        futr            = futr,
        available_mask  = available_mask,
        context_length  = context_length,
        horizon         = horizon,
        channel_ids     = [f"ch_{i}" for i in range(n_channels)],
        name            = name,
        is_multivariate = is_multivariate,
    )


def _make_loader(datasets_with_meta, batch_size=2, shuffle=True, seed=0):
    """datasets_with_meta: list of (dataset, weight, is_multivariate, horizon)"""
    group_datasets = defaultdict(list)
    group_weights  = defaultdict(list)
    global_offsets = defaultdict(list)
    all_datasets   = []
    flat_offset    = 0

    for ds, weight, is_mv, horizon in datasets_with_meta:
        group_key = (horizon, is_mv)
        global_offsets[group_key].append(flat_offset)
        group_datasets[group_key].append(ds)
        group_weights[group_key].append(weight)
        all_datasets.append(ds)
        flat_offset += len(ds)

    combined = ConcatDataset(all_datasets)
    sampler  = HorizonBatchSampler(
        group_datasets = group_datasets,
        group_weights  = group_weights,
        global_offsets = global_offsets,
        batch_size     = batch_size,
        shuffle        = shuffle,
        drop_last      = False,
        seed           = seed,
    )
    return DataLoader(combined, batch_sampler=sampler, collate_fn=_full_series_collate_fn), sampler


def check(name, condition, msg=""):
    status = "✅ PASS" if condition else f"❌ FAIL  {msg}"
    print(f"  {status}  —  {name}")

# %%
# ── Test 1: univariate loader — is_multivariate always False ─────────────────

uni_ds = _make_dataset(1, is_multivariate=False, name="uni")
loader, _ = _make_loader([(uni_ds, 1.0, False, 24)])

failed = [b["is_multivariate"] for b in loader if b["is_multivariate"] is not False]
check("univariate loader → is_multivariate=False", len(failed) == 0,
      f"{len(failed)} batches had is_multivariate=True")

# %%
# ── Test 2: multivariate loader — is_multivariate always True ────────────────

mv_ds = _make_dataset(5, is_multivariate=True, name="mv")
loader, _ = _make_loader([(mv_ds, 1.0, True, 24)])

failed = [b["is_multivariate"] for b in loader if b["is_multivariate"] is not True]
check("multivariate loader → is_multivariate=True", len(failed) == 0,
      f"{len(failed)} batches had is_multivariate=False")

# %%
# ── Test 3: mixed loader — no batch contains both types ──────────────────────

uni_ds = _make_dataset(1, is_multivariate=False, name="uni")
mv_ds  = _make_dataset(5, is_multivariate=True,  name="mv")
loader, _ = _make_loader([
    (uni_ds, 1.0, False, 24),
    (mv_ds,  1.0, True,  24),
], batch_size=2)

mixed_batches = []
for batch in loader:
    flag  = batch["is_multivariate"]
    names = batch["dataset_name"]
    if any((name == "mv") != flag for name in names):
        mixed_batches.append(names)

check("mixed loader → no batch mixes uni + mv", len(mixed_batches) == 0,
      f"{len(mixed_batches)} mixed batches found: {mixed_batches}")

# %%
# ── Test 4: same horizon, different modality — still separated ───────────────

uni_h24 = _make_dataset(1, horizon=24, is_multivariate=False, name="uni_h24")
mv_h24  = _make_dataset(5, horizon=24, is_multivariate=True,  name="mv_h24")
loader, _ = _make_loader([
    (uni_h24, 1.0, False, 24),
    (mv_h24,  1.0, True,  24),
], batch_size=2)

seen_groups   = set()
mixed_batches = []
for batch in loader:
    seen_groups.add((int(batch["horizon"][0]), batch["is_multivariate"]))
    flag  = batch["is_multivariate"]
    names = batch["dataset_name"]
    if any((name == "mv_h24") != flag for name in names):
        mixed_batches.append(names)

check("same horizon, diff modality → not mixed",     len(mixed_batches) == 0,
      f"{len(mixed_batches)} mixed batches")
check("both groups (24,False) and (24,True) appear", {(24, False), (24, True)}.issubset(seen_groups),
      f"seen only: {seen_groups}")

# %%
# ── Test 5: separation holds across multiple epochs ──────────────────────────

uni_ds = _make_dataset(1, is_multivariate=False, name="uni")
mv_ds  = _make_dataset(5, is_multivariate=True,  name="mv")
loader, sampler = _make_loader([
    (uni_ds, 1.0, False, 24),
    (mv_ds,  1.0, True,  24),
], batch_size=2, shuffle=True)

epoch_failures = []
for epoch in range(5):
    sampler.set_epoch(epoch)
    for batch in loader:
        flag  = batch["is_multivariate"]
        names = batch["dataset_name"]
        if any((name == "mv") != flag for name in names):
            epoch_failures.append((epoch, names))

check("separation holds across 5 epochs", len(epoch_failures) == 0,
      f"{len(epoch_failures)} failures: {epoch_failures}")

# %%
# ── Test 6: _dataset_entry defaults and propagation ──────────────────────────

entry_no_flag = _dataset_entry({
    "path": "dummy.csv", "name": "dummy",
    "horizon": 24, "val_size": 100, "test_size": 100,
})
check("_dataset_entry defaults multivariate=False",
      entry_no_flag.multivariate is False,
      f"got {entry_no_flag.multivariate}")

entry_with_flag = _dataset_entry({
    "path": "dummy.csv", "name": "dummy",
    "horizon": 24, "val_size": 100, "test_size": 100,
    "multivariate": True,
})
check("_dataset_entry propagates multivariate=True",
      entry_with_flag.multivariate is True,
      f"got {entry_with_flag.multivariate}")